# Task 1

In [1]:
import os

print(os.getcwd())

In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_json("data/Sarcasm_Headlines_Dataset_v2.json", lines=True)

student_id = 277227
seed = student_id % 100000
np.random.seed(seed)

#Smoothing set based on seed
if seed % 3 == 0:
    alpha_set = [0, 0.1, 1]
elif seed % 3 == 1:
    alpha_set = [0, 0.5, 2]
else:
    alpha_set = [0, 1, 5]

#Ablation index
k = seed % 4

print("Seed:", seed)
print("Smoothing set:", alpha_set)
print("Ablation index:", k)


Seed: 77227
Smoothing set: [0, 0.5, 2]
Ablation index: 3


In [3]:
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [4]:
def preprocess_text(text):
    if not isinstance(text, str):
        return []

    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'!{2,}', ' [EXC_SEQ] ', text)
    text = re.sub(r'\?{2,}', ' [QUE_SEQ] ', text)
    text = re.sub(r'(\?!|!\?)', ' [INT_MARK] ', text)
    text = re.sub(r'\.\.\.+', ' [ELLIP] ', text)
    text = re.sub(r"[^\w\s']+", '', text)
    tokens = re.findall(r"\b\w+'\w+|\w+\b|\[EXC_SEQ\]|\[QUE_SEQ\]|\[INT_MARK\]|\[ELLIP\]", text)
    return tokens

df['tokens'] = df['headline'].apply(preprocess_text)

In [5]:
df.head()

,is_sarcastic,headline,article_link,tokens
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...,"[thirtysomething, scientists, unveil, doomsday..."
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...,"[dem, rep, totally, nails, why, congress, is, ..."
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...,"[eat, your, veggies, 9, deliciously, different..."
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...,"[inclement, weather, prevents, liar, from, get..."
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...,"[mother, comes, pretty, close, to, using, word..."


In [6]:
def handle_negation(tokens):
    """
    Prefix the next N words with NOT_ if a negation word is found.
    N = number of vowels in the negation word.
    """
    negations = {"not","no","never","don't","didn't","isn't","wasn't"}
    vowels = set("aeiou")
    
    result = []
    skip = 0 
    
    for token in tokens:
        if skip > 0:
            result.append("NOT_" + token)
            skip -= 1
        elif token in negations:
            vowel_count = sum(1 for c in token if c in vowels)
            skip = vowel_count
            result.append(token)  
        else:
            result.append(token)
    
    return result

df['tokens_neg'] = df['tokens'].apply(handle_negation)

print(df[['tokens','tokens_neg']].head())

                                              tokens  \
0  [thirtysomething, scientists, unveil, doomsday...   
1  [dem, rep, totally, nails, why, congress, is, ...   
2  [eat, your, veggies, 9, deliciously, different...   
3  [inclement, weather, prevents, liar, from, get...   
4  [mother, comes, pretty, close, to, using, word...   

                                          tokens_neg  
0  [thirtysomething, scientists, unveil, doomsday...  
1  [dem, rep, totally, nails, why, congress, is, ...  
2  [eat, your, veggies, 9, deliciously, different...  
3  [inclement, weather, prevents, liar, from, get...  
4  [mother, comes, pretty, close, to, using, word...  


In [7]:
df.head()

,is_sarcastic,headline,article_link,tokens,tokens_neg
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...,"[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday..."
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...,"[dem, rep, totally, nails, why, congress, is, ...","[dem, rep, totally, nails, why, congress, is, ..."
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...,"[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different..."
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...,"[inclement, weather, prevents, liar, from, get...","[inclement, weather, prevents, liar, from, get..."
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...,"[mother, comes, pretty, close, to, using, word...","[mother, comes, pretty, close, to, using, word..."


In [8]:
stopwords = {
    "a","an","the","and","or","but","if","while","is","am","are","was","were",
    "be","been","being","do","does","did","have","has","had",
    "i","you","he","she","it","we","they","this","that","these","those",
    "in","on","at","for","with","about","to","from","by","of","as"
}

def remove_stopwords(tokens):
    """
    Removing stopwords unless the token starts with NOT_
    """
    return [token for token in tokens if token not in stopwords or token.startswith("NOT_")]

df['tokens_final'] = df['tokens_neg'].apply(remove_stopwords)

print(df[['tokens_neg','tokens_final']].head())

                                          tokens_neg  \
0  [thirtysomething, scientists, unveil, doomsday...   
1  [dem, rep, totally, nails, why, congress, is, ...   
2  [eat, your, veggies, 9, deliciously, different...   
3  [inclement, weather, prevents, liar, from, get...   
4  [mother, comes, pretty, close, to, using, word...   

                                        tokens_final  
0  [thirtysomething, scientists, unveil, doomsday...  
1  [dem, rep, totally, nails, why, congress, fall...  
2  [eat, your, veggies, 9, deliciously, different...  
3  [inclement, weather, prevents, liar, getting, ...  
4  [mother, comes, pretty, close, using, word, st...  


In [9]:
df.head()

,is_sarcastic,headline,article_link,tokens,tokens_neg,tokens_final
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...,"[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday..."
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...,"[dem, rep, totally, nails, why, congress, is, ...","[dem, rep, totally, nails, why, congress, is, ...","[dem, rep, totally, nails, why, congress, fall..."
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...,"[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different..."
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...,"[inclement, weather, prevents, liar, from, get...","[inclement, weather, prevents, liar, from, get...","[inclement, weather, prevents, liar, getting, ..."
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...,"[mother, comes, pretty, close, to, using, word...","[mother, comes, pretty, close, to, using, word...","[mother, comes, pretty, close, using, word, st..."


In [10]:
def generate_ngrams(tokens):
    """
    Generating unigrams and bigrams 
    """
    unigrams = tokens 

    bigrams = [tokens[i] + "_" + tokens[i+1] for i in range(len(tokens)-1)]
    
    return unigrams + bigrams

df['features'] = df['tokens_final'].apply(generate_ngrams)


print(df[['tokens_final','features']].head())

                                        tokens_final  \
0  [thirtysomething, scientists, unveil, doomsday...   
1  [dem, rep, totally, nails, why, congress, fall...   
2  [eat, your, veggies, 9, deliciously, different...   
3  [inclement, weather, prevents, liar, getting, ...   
4  [mother, comes, pretty, close, using, word, st...   

                                            features  
0  [thirtysomething, scientists, unveil, doomsday...  
1  [dem, rep, totally, nails, why, congress, fall...  
2  [eat, your, veggies, 9, deliciously, different...  
3  [inclement, weather, prevents, liar, getting, ...  
4  [mother, comes, pretty, close, using, word, st...  


In [11]:
df.head()

,is_sarcastic,headline,article_link,tokens,tokens_neg,tokens_final,features
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...,"[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday...","[thirtysomething, scientists, unveil, doomsday..."
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...,"[dem, rep, totally, nails, why, congress, is, ...","[dem, rep, totally, nails, why, congress, is, ...","[dem, rep, totally, nails, why, congress, fall...","[dem, rep, totally, nails, why, congress, fall..."
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...,"[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different...","[eat, your, veggies, 9, deliciously, different..."
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...,"[inclement, weather, prevents, liar, from, get...","[inclement, weather, prevents, liar, from, get...","[inclement, weather, prevents, liar, getting, ...","[inclement, weather, prevents, liar, getting, ..."
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...,"[mother, comes, pretty, close, to, using, word...","[mother, comes, pretty, close, to, using, word...","[mother, comes, pretty, close, using, word, st...","[mother, comes, pretty, close, using, word, st..."


In [12]:

df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

# (70% train, 15% validation, 15% test)
n = len(df)
train_df = df[:int(0.7*n)].copy()
validation_df = df[int(0.7*n):int(0.85*n)].copy()
test_df = df[int(0.85*n):].copy()

vocab = set()
for tokens in train_df['features']:
    vocab.update(tokens)

# Replace out-of-vocabulary tokens in validation and test sets
def replace_oov(tokens, vocab):
    return [token if token in vocab else "<UNK>" for token in tokens]

validation_df['features'] = validation_df['features'].apply(lambda x: replace_oov(x, vocab))
test_df['features'] = test_df['features'].apply(lambda x: replace_oov(x, vocab))

print("Train size:", len(train_df))
print("Validation size:", len(validation_df))
print("Test size:", len(test_df))
print("Example validation features:", validation_df['features'].head(1).values[0])

Train size: 20033
Validation size: 4293
Test size: 4293
Example validation features: ['michael', 'jackson', 'hires', 'magical', '<UNK>', '<UNK>', 'defense', 'lawyer', 'michael_jackson', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']


# Task 2

In [13]:
from collections import Counter

# Counting tokens 
df_counter = Counter()
for tokens in train_df['tokens_final']:  
    unique_tokens_in_doc = set(tokens)  
    df_counter.update(unique_tokens_in_doc)

In [14]:
t = (seed % 6) + 2  # threshold
filtered_tokens = [token for token, count in df_counter.items() if count >= t]

In [15]:
filtered_tokens.sort()  

In [16]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5)+1):
        if n % i == 0:
            return False
    return True

prime_tokens = {}     
non_prime_tokens = [] 

for idx, token in enumerate(filtered_tokens, start=1):  
    if is_prime(idx):
        prime_tokens[token] = idx  
    else:
        non_prime_tokens.append(token)

In [17]:
num_docs = len(train_df)


prime_token_to_col = prime_tokens 

num_features = max(prime_token_to_col.values(), default=0) + 1  

dt_matrix = np.zeros((num_docs, num_features), dtype=int)


for i, tokens in enumerate(train_df['tokens_final']):  
    dt_matrix[i, 0] = sum(1 for token in tokens if token in non_prime_tokens)
    for token in tokens:
        if token in prime_token_to_col:
            col_idx = prime_token_to_col[token] 
            dt_matrix[i, col_idx] += 1

In [18]:
top_10_df = df_counter.most_common(10)
print("Top 10 tokens by Document Frequency:")
for token, count in top_10_df:
    print(f"{token}: {count}")

Top 10 tokens by Document Frequency:
new: 1185
trump: 986
man: 951
after: 735
out: 643
how: 599
up: 580
not: 551
what: 478
just: 470


In [19]:
#b)

In [20]:
doc_lengths = dt_matrix.sum(axis=1, keepdims=True)

doc_lengths[doc_lengths == 0] = 1

# TF = term count / total terms in document
tf_matrix = dt_matrix / doc_lengths

In [21]:
import math

num_features = dt_matrix.shape[1]
idf = np.zeros(num_features)

idf[0] = 0

for token, col_idx in prime_token_to_col.items():  # use actual prime column indices
    df_count = df_counter[token]
    idf[col_idx] = math.log(num_docs / df_count)

In [22]:
# Element-wise multiplication
tfidf_matrix = tf_matrix * idf

In [23]:
#c)

In [24]:
from collections import Counter

df_counter = Counter()
for tokens in train_df['tokens_final']:  
    df_counter.update(set(tokens))       

# top 10 tokens 
top_10_df = df_counter.most_common(10)

print("Top 10 tokens by Document Frequency in training set:")
for token, count in top_10_df:
    print(f"{token}: {count}")

Top 10 tokens by Document Frequency in training set:
new: 1185
trump: 986
man: 951
after: 735
out: 643
how: 599
up: 580
not: 551
what: 478
just: 470


# Task 3

In [25]:
#multinomial

In [26]:
import numpy as np

y_train = train_df['is_sarcastic'].values

# Class priors
prior_sarc = np.mean(y_train == 1)
prior_not = np.mean(y_train == 0)

log_prior_sarc = np.log(prior_sarc)
log_prior_not = np.log(prior_not)

print("Log Prior Sarcastic:", log_prior_sarc)
print("Log Prior Not Sarcastic:", log_prior_not)

Log Prior Sarcastic: -0.7420970939234828
Log Prior Not Sarcastic: -0.6464819605269192


In [27]:
from collections import defaultdict

vocab = set()
for tokens in train_df['features']:
    vocab.update(tokens)
vocab = list(vocab)
V = len(vocab)
print("Vocabulary size:", V)

word_counts_sarc = defaultdict(int)
word_counts_not = defaultdict(int)
total_words_sarc = 0
total_words_not = 0

for tokens, label in zip(train_df['features'], y_train):
    if label == 1:
        for token in tokens:
            word_counts_sarc[token] += 1
            total_words_sarc += 1
    else:
        for token in tokens:
            word_counts_not[token] += 1
            total_words_not += 1

Vocabulary size: 144462


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score

vocab_list = list(vocab)
token_to_idx = {token: i for i, token in enumerate(vocab_list)}
V = len(vocab_list)

word_counts_sarc_arr = np.array([word_counts_sarc.get(token, 0) for token in vocab_list])
word_counts_not_arr  = np.array([word_counts_not.get(token, 0) for token in vocab_list])

val_matrix = np.zeros((len(validation_df), V))
for i, tokens in enumerate(validation_df['features']):
    for t in tokens:
        if t in token_to_idx:
            val_matrix[i, token_to_idx[t]] += 1

best_alpha = None
best_acc = -1

for alpha_val in alpha_set:
    a = alpha_val if alpha_val > 0 else 1e-10

    ll_sarc_arr = np.log((word_counts_sarc_arr + a) / (total_words_sarc + a * V))
    ll_not_arr  = np.log((word_counts_not_arr  + a) / (total_words_not  + a * V))

    log_probs_sarc = log_prior_sarc + val_matrix.dot(ll_sarc_arr)
    log_probs_not  = log_prior_not  + val_matrix.dot(ll_not_arr)

    preds = (log_probs_sarc > log_probs_not).astype(int)

    acc = accuracy_score(validation_df['is_sarcastic'], preds)
    print(f"Alpha={alpha_val} → Validation Accuracy: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_alpha = alpha_val
        log_likelihood_sarc = {token: ll for token, ll in zip(vocab_list, ll_sarc_arr)}
        log_likelihood_not  = {token: ll for token, ll in zip(vocab_list, ll_not_arr)}

alpha = best_alpha if best_alpha > 0 else 1e-10
print(f"\nBest alpha: {best_alpha} with accuracy {best_acc:.4f}")

Alpha=0 → Validation Accuracy: 0.7890
Alpha=0.5 → Validation Accuracy: 0.8341
Alpha=2 → Validation Accuracy: 0.8369

Best alpha: 2 with accuracy 0.8369


In [29]:
def predict_multinomial(tokens):
    
    log_prob_sarc = log_prior_sarc
    log_prob_not = log_prior_not

    for token in tokens:
        if token in vocab: 
            log_prob_sarc += log_likelihood_sarc[token]
            log_prob_not += log_likelihood_not[token]

    return 1 if log_prob_sarc > log_prob_not else 0

In [30]:
from sklearn.metrics import accuracy_score

y_pred_multinomial = [predict_multinomial(tokens) for tokens in validation_df['features']]

acc_multinomial = accuracy_score(validation_df['is_sarcastic'], y_pred_multinomial)
print("Multinomial Validation Accuracy:", acc_multinomial)

Multinomial Validation Accuracy: 0.8369438621010948


In [31]:
#bernoulli

In [32]:
train_sets = [set(tokens) for tokens in train_df['features']]
val_sets = [set(tokens) for tokens in validation_df['features']]

num_docs_sarc = np.sum(y_train == 1)
num_docs_not = np.sum(y_train == 0)

doc_counts_sarc = defaultdict(int)
doc_counts_not = defaultdict(int)

for s, label in zip(train_sets, y_train):
    for token in s:
        if label == 1:
            doc_counts_sarc[token] += 1
        else:
            doc_counts_not[token] += 1

log_present_sarc = {}
log_absent_sarc = {}
log_present_not = {}
log_absent_not = {}

for token in vocab:
    p_sarc = (doc_counts_sarc.get(token, 0) + alpha) / (num_docs_sarc + 2*alpha)
    p_not = (doc_counts_not.get(token, 0) + alpha) / (num_docs_not + 2*alpha)
    
    log_present_sarc[token] = np.log(p_sarc)
    log_absent_sarc[token] = np.log(1 - p_sarc)
    
    log_present_not[token] = np.log(p_not)
    log_absent_not[token] = np.log(1 - p_not)

In [33]:
def bernoulli_likelihood(word, doc_counts, num_docs, alpha):
    count = doc_counts.get(word, 0)
    prob_present = (count + alpha) / (num_docs + 2*alpha)
    prob_present = min(max(prob_present, 1e-12), 1-1e-12) 
    return np.log(prob_present), np.log(1 - prob_present)

In [34]:
def predict_bernoulli(tokens):
    token_set = set(tokens)
    log_prob_sarc = log_prior_sarc
    log_prob_not = log_prior_not

    for token in vocab:
        if token in token_set:
            log_prob_sarc += log_present_sarc[token]
            log_prob_not += log_present_not[token]
        else:
            log_prob_sarc += log_absent_sarc[token]
            log_prob_not += log_absent_not[token]

    return 1 if log_prob_sarc > log_prob_not else 0

In [35]:
from sklearn.metrics import accuracy_score

y_pred_bernoulli = [predict_bernoulli(tokens) for tokens in validation_df['features']]

acc_bernoulli = accuracy_score(validation_df['is_sarcastic'], y_pred_bernoulli)
print("Bernoulli Validation Accuracy:", acc_bernoulli)

Bernoulli Validation Accuracy: 0.6922897740507803


# Task 4

In [36]:
import pandas as pd
import re
import numpy as np

negation_words = {"not", "no", "never", "don't", "didn't", "isn't", "wasn't"}

def extract_numeric_features(row):
    text = str(row['headline']) 
    words = [re.sub(r'\W+', '', w) for w in text.split() if w.strip()]  # clean words

    Total_Word_Count = len(words)
    Char_Count = len(text)
    Uppercase_Count = sum(1 for w in words if w.isupper())
    Exclamation_Count = text.count('!')
    Question_Count = text.count('?')
    Negation_Count = sum(1 for w in words if w.lower() in negation_words)
    Digit_Count = sum(c.isdigit() for c in text)

    Uppercase_Ratio = Uppercase_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Exclamation_Ratio = Exclamation_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Question_Ratio = Question_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Digit_Ratio = Digit_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Avg_Word_Length = Char_Count / Total_Word_Count if Total_Word_Count > 0 else 0

    Sarcasm_Ratio = Uppercase_Ratio if Total_Word_Count > 0 else 0.5

    return pd.Series([
        Total_Word_Count, Char_Count, Uppercase_Count, Uppercase_Ratio,
        Exclamation_Count, Exclamation_Ratio, Question_Count, Question_Ratio,
        Negation_Count, Digit_Count, Digit_Ratio, Avg_Word_Length, Sarcasm_Ratio
    ])

numeric_cols = [
    'Total_Word_Count', 'Char_Count', 'Uppercase_Count', 'Uppercase_Ratio',
    'Exclamation_Count', 'Exclamation_Ratio', 'Question_Count', 'Question_Ratio',
    'Negation_Count', 'Digit_Count', 'Digit_Ratio', 'Avg_Word_Length', 'Sarcasm_Ratio'
]

df[numeric_cols] = df.apply(extract_numeric_features, axis=1)

df[numeric_cols] = df[numeric_cols].fillna(0)

print(df[numeric_cols].head())

   Total_Word_Count  Char_Count  Uppercase_Count  Uppercase_Ratio  \
0              14.0        79.0              0.0              0.0   
1              12.0        66.0              0.0              0.0   
2              13.0        75.0              0.0              0.0   
3              11.0        71.0              0.0              0.0   
4              12.0        81.0              0.0              0.0   

   Exclamation_Count  Exclamation_Ratio  Question_Count  Question_Ratio  \
0                0.0                0.0             0.0             0.0   
1                0.0                0.0             0.0             0.0   
2                0.0                0.0             0.0             0.0   
3                0.0                0.0             0.0             0.0   
4                0.0                0.0             0.0             0.0   

   Negation_Count  Digit_Count  Digit_Ratio  Avg_Word_Length  Sarcasm_Ratio  
0             0.0          0.0     0.000000         5.64

In [37]:
#a

In [38]:
np.random.seed(seed)

for col in numeric_cols:
    n_missing = max(1, int(0.05 * len(df)))
    missing_idx = np.random.choice(df.index, size=n_missing, replace=False)
    df.loc[missing_idx, col] = np.nan

if df['Sarcasm_Ratio'].isna().sum() == 0:
    df.loc[df.index[0], 'Sarcasm_Ratio'] = np.nan

print("Missing values injected per column:")
print(df[numeric_cols].isna().sum())

Missing values injected per column:
Total_Word_Count     1430
Char_Count           1430
Uppercase_Count      1430
Uppercase_Ratio      1430
Exclamation_Count    1430
Exclamation_Ratio    1430
Question_Count       1430
Question_Ratio       1430
Negation_Count       1430
Digit_Count          1430
Digit_Ratio          1430
Avg_Word_Length      1430
Sarcasm_Ratio        1430
dtype: int64


In [39]:
#b

In [40]:
df_mean_imputed = df.copy()

for col in numeric_cols:
    mean_value = df_mean_imputed[col].mean()
    df_mean_imputed[col] = df_mean_imputed[col].fillna(mean_value)

df_class_imputed = df.copy()

for col in numeric_cols:
    class_means = df_class_imputed.groupby('is_sarcastic')[col].transform('mean')
    df_class_imputed[col] = df_class_imputed[col].fillna(class_means)

print("Missing values after mean imputation:\n", df_mean_imputed[numeric_cols].isna().sum())
print("\nMissing values after class-conditional mean imputation:\n", df_class_imputed[numeric_cols].isna().sum())

Missing values after mean imputation:
 Total_Word_Count     0
Char_Count           0
Uppercase_Count      0
Uppercase_Ratio      0
Exclamation_Count    0
Exclamation_Ratio    0
Question_Count       0
Question_Ratio       0
Negation_Count       0
Digit_Count          0
Digit_Ratio          0
Avg_Word_Length      0
Sarcasm_Ratio        0
dtype: int64

Missing values after class-conditional mean imputation:
 Total_Word_Count     0
Char_Count           0
Uppercase_Count      0
Uppercase_Ratio      0
Exclamation_Count    0
Exclamation_Ratio    0
Question_Count       0
Question_Ratio       0
Negation_Count       0
Digit_Count          0
Digit_Ratio          0
Avg_Word_Length      0
Sarcasm_Ratio        0
dtype: int64


# Task 5

In [41]:
numeric_cols = [
    'Total_Word_Count', 'Char_Count', 'Uppercase_Count', 'Uppercase_Ratio',
    'Exclamation_Count', 'Exclamation_Ratio', 'Question_Count', 'Question_Ratio',
    'Negation_Count', 'Digit_Count', 'Digit_Ratio', 'Avg_Word_Length', 'Sarcasm_Ratio'
]

print("Outlier Detection (IQR method):")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f"{col}: Lower Bound = {lower_bound}, Upper Bound = {upper_bound}")

Outlier Detection (IQR method):
Total_Word_Count: Lower Bound = 2.0, Upper Bound = 18.0
Char_Count: Lower Bound = 10.0, Upper Bound = 114.0
Uppercase_Count: Lower Bound = 0.0, Upper Bound = 0.0
Uppercase_Ratio: Lower Bound = 0.0, Upper Bound = 0.0
Exclamation_Count: Lower Bound = 0.0, Upper Bound = 0.0
Exclamation_Ratio: Lower Bound = 0.0, Upper Bound = 0.0
Question_Count: Lower Bound = 0.0, Upper Bound = 0.0
Question_Ratio: Lower Bound = 0.0, Upper Bound = 0.0
Negation_Count: Lower Bound = 0.0, Upper Bound = 0.0
Digit_Count: Lower Bound = 0.0, Upper Bound = 0.0
Digit_Ratio: Lower Bound = 0.0, Upper Bound = 0.0
Avg_Word_Length: Lower Bound = 3.8125000000000004, Upper Bound = 8.645833333333332
Sarcasm_Ratio: Lower Bound = 0.0, Upper Bound = 0.0


In [42]:
df_clamped = df.copy()

for col in numeric_cols:
    Q1 = df_clamped[col].quantile(0.25)
    Q3 = df_clamped[col].quantile(0.75)
    IQR = Q3 - Q1

    if IQR > 0:
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clamped[col] = df_clamped[col].clip(lower_bound, upper_bound)

print("Class distribution before clamping:\n", df['is_sarcastic'].value_counts())
print("\nClass distribution after clamping:\n", df_clamped['is_sarcastic'].value_counts())

print("\nMin/Max of numeric features after clamping:")
print(df_clamped[numeric_cols].agg(['min','max']))

Class distribution before clamping:
 is_sarcastic
0    14985
1    13634
Name: count, dtype: int64

Class distribution after clamping:
 is_sarcastic
0    14985
1    13634
Name: count, dtype: int64

Min/Max of numeric features after clamping:
     Total_Word_Count  Char_Count  Uppercase_Count  Uppercase_Ratio  \
min               2.0        10.0              0.0              0.0   
max              18.0       114.0              0.0              0.0   

     Exclamation_Count  Exclamation_Ratio  Question_Count  Question_Ratio  \
min                0.0                0.0             0.0        0.000000   
max                3.0                0.5             2.0        0.666667   

     Negation_Count  Digit_Count  Digit_Ratio  Avg_Word_Length  Sarcasm_Ratio  
min             0.0          0.0          0.0         3.812500            0.0  
max             3.0         18.0          2.0         8.645833            0.0  


# Task 6

In [43]:
# 6A 
numeric_cols = [
    'Total_Word_Count', 'Char_Count', 'Uppercase_Count', 'Uppercase_Ratio',
    'Exclamation_Count', 'Exclamation_Ratio', 'Question_Count', 'Question_Ratio',
    'Negation_Count', 'Digit_Count', 'Digit_Ratio', 'Avg_Word_Length', 'Sarcasm_Ratio'
]
for split_df in [train_df, validation_df, test_df]:
    split_df[numeric_cols] = split_df.apply(extract_numeric_features, axis=1)

print("Train numeric features head:")
print(train_df[numeric_cols].head())

Train numeric features head:
   Total_Word_Count  Char_Count  Uppercase_Count  Uppercase_Ratio  \
0              14.0        79.0              0.0              0.0   
1              12.0        66.0              0.0              0.0   
2              13.0        75.0              0.0              0.0   
3              11.0        71.0              0.0              0.0   
4              12.0        81.0              0.0              0.0   

   Exclamation_Count  Exclamation_Ratio  Question_Count  Question_Ratio  \
0                0.0                0.0             0.0             0.0   
1                0.0                0.0             0.0             0.0   
2                0.0                0.0             0.0             0.0   
3                0.0                0.0             0.0             0.0   
4                0.0                0.0             0.0             0.0   

   Negation_Count  Digit_Count  Digit_Ratio  Avg_Word_Length  Sarcasm_Ratio  
0             0.0          

In [44]:
# 6B 

from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

X_train = train_df[numeric_cols].values
y_train = train_df['is_sarcastic'].values

X_val = validation_df[numeric_cols].values
y_val = validation_df['is_sarcastic'].values

minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_val_minmax = minmax_scaler.transform(X_val)

gnb_minmax = GaussianNB()
gnb_minmax.fit(X_train_minmax, y_train)

y_pred_minmax = gnb_minmax.predict(X_val_minmax)
acc_minmax = accuracy_score(y_val, y_pred_minmax)
print("Validation Accuracy with MinMax Scaling:", acc_minmax)

Validation Accuracy with MinMax Scaling: 0.49732122059166084


In [45]:
# 6C 

from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_val_std = std_scaler.transform(X_val)

gnb_std = GaussianNB()
gnb_std.fit(X_train_std, y_train)

y_pred_std = gnb_std.predict(X_val_std)
acc_std = accuracy_score(y_val, y_pred_std)
print("Validation Accuracy with Standardisation:", acc_std)

Validation Accuracy with Standardisation: 0.49732122059166084


# Task 7

In [46]:
print("Class distribution before resampling:")
print(train_df['is_sarcastic'].value_counts())

Class distribution before resampling:
is_sarcastic
0    10495
1     9538
Name: count, dtype: int64


In [47]:
from sklearn.utils import resample

df_majority = train_df[train_df['is_sarcastic'] == 0]
df_minority = train_df[train_df['is_sarcastic'] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,               
    n_samples=len(df_majority), 
    random_state=seed
)

train_resampled = pd.concat([df_majority, df_minority_upsampled])

train_resampled = train_resampled.sample(frac=1, random_state=seed).reset_index(drop=True)

print("\nClass distribution after oversampling:")
print(train_resampled['is_sarcastic'].value_counts())


Class distribution after oversampling:
is_sarcastic
0    10495
1    10495
Name: count, dtype: int64


In [48]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

numeric_cols = [
    'Total_Word_Count', 'Char_Count', 'Uppercase_Count', 'Uppercase_Ratio',
    'Exclamation_Count', 'Exclamation_Ratio', 'Question_Count', 'Question_Ratio',
    'Negation_Count', 'Digit_Count', 'Digit_Ratio', 'Avg_Word_Length', 'Sarcasm_Ratio'
]

X_train_resampled = train_resampled[numeric_cols].values
y_train_resampled = train_resampled['is_sarcastic'].values

X_val = validation_df[numeric_cols].values
y_val = validation_df['is_sarcastic'].values

gnb = GaussianNB()
gnb.fit(X_train_resampled, y_train_resampled)

y_probs = gnb.predict_proba(X_val)[:, 1]

y_pred = (y_probs >= 0.5).astype(int)

cm = confusion_matrix(y_val, y_pred) 

TP = cm[1,1]
FP = cm[0,1]
FN = cm[1,0]
TN = cm[0,0]

print("Confusion Matrix (TP, FP, FN, TN):")
print(f"TP: {TP}, FP: {FP}, FN: {FN}, TN: {TN}")

precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1_macro = f1_score(y_val, y_pred, average='macro')
f1_minority = f1_score(y_val, y_pred, pos_label=1)  # minority class
accuracy = accuracy_score(y_val, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Minority Class): {f1_minority:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Confusion Matrix (TP, FP, FN, TN):
TP: 1989, FP: 2142, FN: 16, TN: 146
Precision: 0.4815
Recall: 0.9920
F1 Score (Macro): 0.3837
F1 Score (Minority Class): 0.6483
Accuracy: 0.4973


In [49]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Macro':<12} {'F1 Minority'}")
for thresh in thresholds:
    y_pred_t = (y_probs >= thresh).astype(int)
    print(f"{thresh:<12} "
          f"{accuracy_score(y_val, y_pred_t):<12.4f} "
          f"{precision_score(y_val, y_pred_t, zero_division=0):<12.4f} "
          f"{recall_score(y_val, y_pred_t, zero_division=0):<12.4f} "
          f"{f1_score(y_val, y_pred_t, average='macro'):<12.4f} "
          f"{f1_score(y_val, y_pred_t, pos_label=1, zero_division=0):.4f}")

Threshold    Accuracy     Precision    Recall       F1 Macro     F1 Minority
0.3          0.4973       0.4815       0.9920       0.3837       0.6483
0.4          0.4973       0.4815       0.9920       0.3837       0.6483
0.5          0.4973       0.4815       0.9920       0.3837       0.6483
0.6          0.4973       0.4815       0.9920       0.3837       0.6483
0.7          0.4973       0.4815       0.9920       0.3837       0.6483


# Task 8

In [50]:
X_test = test_df[numeric_cols].values
y_test = test_df['is_sarcastic'].values

y_test_probs = gnb.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_probs >= 0.5).astype(int)

misclassified_idx = np.where(y_test_pred != y_test)[0]

print("8 Misclassified Headlines:")
for idx in misclassified_idx[:8]:
    print(f"Headline: {test_df.iloc[idx]['headline']}")
    print(f"True Label: {y_test[idx]}, Predicted: {y_test_pred[idx]}\n")

8 Misclassified Headlines:
Headline: fda warns another company about unapproved consumer genetic tests
True Label: 0, Predicted: 1

Headline: documentary review: 3 1/2 minutes, ten bullets
True Label: 0, Predicted: 1

Headline: the 10 busiest days for summer travel
True Label: 0, Predicted: 1

Headline: rex tillerson can expect a lot of questions about his record on climate change
True Label: 0, Predicted: 1

Headline: michael bay might secretly be a genius, despite his awful movies
True Label: 0, Predicted: 1

Headline: billionaire cash is flooding los angeles to push trump-devos school choice agenda
True Label: 0, Predicted: 1

Headline: councilman calls on baltimore rappers to inspire students
True Label: 0, Predicted: 1

Headline: the most important first step to success
True Label: 0, Predicted: 1



In [51]:
new_headlines = [
    "Trump and Iran got compromised" ,  # Sarcastic
    "USA Dollar has crashed",  # Serious
    "Assigngments are completed, Should i worry about the job?"  # Mixed
]

def extract_features_for_new(text, url="example.com"):
    words = [w for w in text.split() if w.strip()]
    Total_Word_Count = len(words)
    Char_Count = len(text)
    Uppercase_Count = sum(1 for w in words if w.isupper())
    Exclamation_Count = text.count('!')
    Question_Count = text.count('?')
    Negation_Count = sum(1 for w in words if w.lower() in negation_words)
    Digit_Count = sum(c.isdigit() for c in text)

    Uppercase_Ratio = Uppercase_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Exclamation_Ratio = Exclamation_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Question_Ratio = Question_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Digit_Ratio = Digit_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Avg_Word_Length = Char_Count / Total_Word_Count if Total_Word_Count > 0 else 0
    Sarcasm_Ratio = Uppercase_Ratio if Total_Word_Count > 0 else 0.5

    return [
        Total_Word_Count, Char_Count, Uppercase_Count, Uppercase_Ratio,
        Exclamation_Count, Exclamation_Ratio, Question_Count, Question_Ratio,
        Negation_Count, Digit_Count, Digit_Ratio, Avg_Word_Length, Sarcasm_Ratio
    ]

X_new = np.array([extract_features_for_new(h) for h in new_headlines])
y_new_probs = gnb.predict_proba(X_new)[:, 1]
y_new_pred = (y_new_probs >= 0.5).astype(int)

for h, pred, prob in zip(new_headlines, y_new_pred, y_new_probs):
    print(f"Headline: {h}")
    print(f"Predicted Label: {pred} (Prob={prob:.2f})\n")

Headline: Trump and Iran got compromised
Predicted Label: 1 (Prob=0.97)

Headline: USA Dollar has crashed
Predicted Label: 1 (Prob=0.98)

Headline: Assigngments are completed, Should i worry about the job?
Predicted Label: 0 (Prob=0.00)

